# 제6장: 거버넌스 도구와 플랫폼

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch06.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy captum

### AI 거버넌스 도구 및 기술

## AI 거버넌스 기술 스택 개요

### 거버넌스 기술 스택 아키텍처

### 도구 카테고리별 개요

### 도구 선정 기준

## 공정성 평가 및 편향 완화 도구

### Fairlearn 심화 활용

**Fairlearn 0.14.0 Advanced Usage**

In [ ]:
"""
Fairlearn 0.14.0 Advanced Usage Guide

- Multi-dimensional fairness analysis using MetricFrame
- Intersectionality analysis for multiple sensitive attributes
- Defining custom metrics
- Comparing bias mitigation algorithms
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
 accuracy_score, precision_score, recall_score, 
 f1_score, roc_auc_score
)

from fairlearn.metrics import (
 MetricFrame,
 demographic_parity_difference,
 demographic_parity_ratio,
 equalized_odds_difference,
 selection_rate,
 true_positive_rate,
 false_positive_rate,
 count,
)
from fairlearn.reductions import (
 ExponentiatedGradient,
 GridSearch,
 DemographicParity,
 EqualizedOdds,
 TruePositiveRateParity,
 ErrorRateParity,
)
from fairlearn.postprocessing import ThresholdOptimizer


class AdvancedFairnessAnalyzer:
 """
 Advanced fairness analyzer based on Fairlearn 0.14.0
 
 Supports multiple sensitive attributes, intersectional analysis, and custom metrics
 """
 
 def __init__(self):
 # Base metric set
 self.base_metrics = {
 'accuracy': accuracy_score,
 'precision': precision_score,
 'recall': recall_score,
 'f1': f1_score,
 'selection_rate': selection_rate,
 'tpr': true_positive_rate,
 'fpr': false_positive_rate,
 'count': count,
 }
 
 def create_intersectional_groups(
 self,
 sensitive_features: pd.DataFrame,
 columns: list,
 ) -> pd.Series:
 """
 Create intersectional groups
 
 Example: gender x age_group -> 'female_young', 'male_senior', etc.
 """
 return sensitive_features[columns].apply(
 lambda row: '_'.join(str(v) for v in row), axis=1
 )
 
 def comprehensive_fairness_analysis(
 self,
 y_true: np.ndarray,
 y_pred: np.ndarray,
 sensitive_features: pd.DataFrame,
 intersectional_columns: list = None,
 ) -> dict:
 """
 Comprehensive fairness analysis
 
 Performs analysis for individual sensitive attributes + intersectional groups
 """
 
 results = {
 'individual_attributes': {},
 'intersectional': {},
 'summary': {},
 }
 
 # 1. Individual sensitive attribute analysis
 for col in sensitive_features.columns:
 mf = MetricFrame(
 metrics=self.base_metrics,
 y_true=y_true,
 y_pred=y_pred,
 sensitive_features=sensitive_features[col],
 )
 
 results['individual_attributes'][col] = {
 'by_group': mf.by_group.to_dict(),
 'overall': mf.overall.to_dict(),
 'difference': mf.difference().to_dict(),
 'ratio': mf.ratio().to_dict(),
 'group_min': mf.group_min().to_dict(),
 'group_max': mf.group_max().to_dict(),
 }
 
 # 2. Intersectional analysis (if specified)
 if intersectional_columns and len(intersectional_columns) >= 2:
 intersectional_groups = self.create_intersectional_groups(
 sensitive_features, intersectional_columns
 )
 
 mf_inter = MetricFrame(
 metrics=self.base_metrics,
 y_true=y_true,
 y_pred=y_pred,
 sensitive_features=intersectional_groups,
 )
 
 results['intersectional'] = {
 'columns': intersectional_columns,
 'by_group': mf_inter.by_group.to_dict(),
 'difference': mf_inter.difference().to_dict(),
 'worst_group': mf_inter.group_min().to_dict(),
 }
 
 # 3. Summary metrics
 results['summary'] = {
 'dp_difference': demographic_parity_difference(
 y_true, y_pred,
 sensitive_features=sensitive_features[sensitive_features.columns[0]]
 ),
 'dp_ratio': demographic_parity_ratio(
 y_true, y_pred,
 sensitive_features=sensitive_features[sensitive_features.columns[0]]
 ),
 'eo_difference': equalized_odds_difference(
 y_true, y_pred,
 sensitive_features=sensitive_features[sensitive_features.columns[0]]
 ),
 }
 
 return results
 
 def compare_mitigation_algorithms(
 self,
 X_train: np.ndarray,
 y_train: np.ndarray,
 sensitive_train: np.ndarray,
 X_test: np.ndarray,
 y_test: np.ndarray,
 sensitive_test: np.ndarray,
 base_estimator,
 constraints: list = None,
 ) -> pd.DataFrame:
 """
 Bias mitigation algorithm comparison
 
 Compare multiple constraints and algorithms to find 
 the optimal fairness-accuracy trade-off
 """
 
 if constraints is None:
 constraints = [
 ('Demographic Parity', DemographicParity()),
 ('Equalized Odds', EqualizedOdds()),
 ('TPR Parity', TruePositiveRateParity()),
 ('Error Rate Parity', ErrorRateParity()),
 ]
 
 results = []
 
 # 1. Baseline (No Mitigation)
 base_model = base_estimator.__class__(base_estimator.get_params())
 base_model.fit(X_train, y_train)
 y_pred_base = base_model.predict(X_test)
 
 results.append({
 'method': 'Baseline (No Mitigation)',
 'accuracy': accuracy_score(y_test, y_pred_base),
 'dp_ratio': demographic_parity_ratio(
 y_test, y_pred_base, sensitive_features=sensitive_test
 ),
 'dp_diff': demographic_parity_difference(
 y_test, y_pred_base, sensitive_features=sensitive_test
 ),
 'eo_diff': equalized_odds_difference(
 y_test, y_pred_base, sensitive_features=sensitive_test
 ),
 })
 
 # 2. Apply ExponentiatedGradient for each constraint
 for name, constraint in constraints:
 try:
 mitigator = ExponentiatedGradient(
 estimator=base_estimator.__class__(base_estimator.get_params()),
 constraints=constraint,
 eps=0.01,
 )
 mitigator.fit(X_train, y_train, sensitive_features=sensitive_train)
 y_pred = mitigator.predict(X_test)
 
 results.append({
 'method': f'ExpGrad + {name}',
 'accuracy': accuracy_score(y_test, y_pred),
 'dp_ratio': demographic_parity_ratio(
 y_test, y_pred, sensitive_features=sensitive_test
 ),
 'dp_diff': demographic_parity_difference(
 y_test, y_pred, sensitive_features=sensitive_test
 ),
 'eo_diff': equalized_odds_difference(
 y_test, y_pred, sensitive_features=sensitive_test
 ),
 })
 except Exception as e:
 results.append({
 'method': f'ExpGrad + {name}',
 'accuracy': None,
 'dp_ratio': None,
 'dp_diff': None,
 'eo_diff': None,
 'error': str(e),
 })
 
 # 3. ThresholdOptimizer (Post-processing)
 try:
 threshold_opt = ThresholdOptimizer(
 estimator=base_model,
 constraints="demographic_parity",
 objective="accuracy_score",
 prefit=True,
 )
 threshold_opt.fit(X_train, y_train, sensitive_features=sensitive_train)
 y_pred_thresh = threshold_opt.predict(X_test, sensitive_features=sensitive_test)
 
 results.append({
 'method': 'ThresholdOptimizer (DP)',
 'accuracy': accuracy_score(y_test, y_pred_thresh),
 'dp_ratio': demographic_parity_ratio(
 y_test, y_pred_thresh, sensitive_features=sensitive_test
 ),
 'dp_diff': demographic_parity_difference(
 y_test, y_pred_thresh, sensitive_features=sensitive_test
 ),
 'eo_diff': equalized_odds_difference(
 y_test, y_pred_thresh, sensitive_features=sensitive_test
 ),
 })
 except Exception as e:
 results.append({
 'method': 'ThresholdOptimizer (DP)',
 'error': str(e),
 })
 
 return pd.DataFrame(results)
 
 def generate_fairness_report_latex(
 self,
 analysis_results: dict,
 model_name: str,
 ) -> str:
 """Fairness Analysis LaTeX Report Generation"""
 
 latex = r"""
\begin{table}[htbp]
\centering
\caption{""" + model_name + r""" Fairness Analysis Results}
\begin{tabularx}{\textwidth}{p{3cm}XXXXX}
\toprule
\textbf{Sensitive Attribute} & \textbf{DP Ratio} & \textbf{DP Diff} & \textbf{EO Diff} & \textbf{4/5 Rule} \\
\midrule
"""
 
 for attr, result in analysis_results.get('individual_attributes', {}).items():
 dp_ratio = result.get('ratio', {}).get('selection_rate', 0)
 dp_diff = result.get('difference', {}).get('selection_rate', 0)
 
 # EO diff is the maximum of TPR and FPR differences
 tpr_diff = result.get('difference', {}).get('tpr', 0)
 fpr_diff = result.get('difference', {}).get('fpr', 0)
 eo_diff = max(abs(tpr_diff) if tpr_diff else 0, 
 abs(fpr_diff) if fpr_diff else 0)
 
 passes = '[PASS]' if dp_ratio and dp_ratio >= 0.8 else '[FAIL]'
 
 latex += f"{attr} & {dp_ratio:.4f} & {dp_diff:.4f} & {eo_diff:.4f} & {passes} \\\\\n"
 
 latex += r"""
\bottomrule
\end{tabularx}
\end{table}
"""
 
 return latex

### AIF360 (AI 공정성 360)

## 설명가능성(설명가능한 AI(설명가능한 AI(XAI))) 도구

### SHAP 심화 활용

**SHAP 0.46.0 Advanced Usage**

In [ ]:
"""
SHAP 0.46.0 Advanced Usage Guide

- Selecting optimal Explainer by model type
- Large-scale data processing
- Customizing visualizations
- Governance report integration
"""

import numpy as np
import pandas as pd
import shap
from typing import Optional, Dict, List, Union, Any
import warnings

class SHAPGovernanceExplainer:
 """
 SHAP-based governance explanation generator
 
 Optimized for model types, handles large-scale data,
 and supports governance report formats
 """
 
 # Model-Explainer 
 EXPLAINER_MAP = {
 'tree': shap.TreeExplainer,
 'linear': shap.LinearExplainer,
 'deep': shap.DeepExplainer,
 'gradient': shap.GradientExplainer,
 'kernel': shap.KernelExplainer,
 }
 
 # sklearn based Model
 TREE_MODELS = [
 'RandomForestClassifier', 'RandomForestRegressor',
 'GradientBoostingClassifier', 'GradientBoostingRegressor',
 'XGBClassifier', 'XGBRegressor',
 'LGBMClassifier', 'LGBMRegressor',
 'CatBoostClassifier', 'CatBoostRegressor',
 'DecisionTreeClassifier', 'DecisionTreeRegressor',
 'ExtraTreesClassifier', 'ExtraTreesRegressor',
 ]
 
 # Model
 LINEAR_MODELS = [
 'LogisticRegression', 'LinearRegression',
 'Ridge', 'Lasso', 'ElasticNet',
 'SGDClassifier', 'SGDRegressor',
 ]
 
 def __init__(
 self,
 model: Any,
 feature_names: Optional[List[str]] = None,
 max_samples_for_kernel: int = 100,
 ):
 """
 Parameters
 ----------
 model : Any
 Model to explain
 feature_names : list, optional
 List of feature names
 max_samples_for_kernel : int
 Max background samples for KernelExplainer
 """
 self.model = model
 self.feature_names = feature_names
 self.max_samples_for_kernel = max_samples_for_kernel
 self.explainer = None
 self.explainer_type = None
 self.background_data = None
 
 def _detect_model_type(self) -> str:
 """Auto-detect model type"""
 
 model_class = type(self.model).__name__
 
 if model_class in self.TREE_MODELS:
 return 'tree'
 elif model_class in self.LINEAR_MODELS:
 return 'linear'
 elif 'keras' in str(type(self.model)).lower():
 return 'deep'
 elif 'torch' in str(type(self.model)).lower():
 return 'gradient'
 else:
 return 'kernel'
 
 def fit(
 self,
 X_background: Union[np.ndarray, pd.DataFrame],
 model_type: Optional[str] = None,
 ) -> 'SHAPGovernanceExplainer':
 """
 Initialize Explainer
 
 Parameters
 ----------
 X_background : array-like
 Background data (used in KernelExplainer, DeepExplainer)
 model_type : str, optional
 One of 'tree', 'linear', 'deep', 'gradient', 'kernel'.
 Auto-detected if None.
 """
 
 self.explainer_type = model_type or self._detect_model_type()
 
 # Data 
 if isinstance(X_background, pd.DataFrame):
 self.background_data = X_background.values
 if self.feature_names is None:
 self.feature_names = X_background.columns.tolist()
 else:
 self.background_data = X_background
 
 # Explainer 
 if self.explainer_type == 'tree':
 self.explainer = shap.TreeExplainer(
 self.model,
 feature_perturbation='interventional',
 )
 
 elif self.explainer_type == 'linear':
 self.explainer = shap.LinearExplainer(
 self.model,
 self.background_data,
 )
 
 elif self.explainer_type in ['deep', 'gradient']:
 # Model: Number of samples 
 bg_sample = self.background_data
 if len(bg_sample) > self.max_samples_for_kernel:
 indices = np.random.choice(
 len(bg_sample), 
 self.max_samples_for_kernel, 
 replace=False
 )
 bg_sample = bg_sample[indices]
 
 if self.explainer_type == 'deep':
 self.explainer = shap.DeepExplainer(self.model, bg_sample)
 else:
 self.explainer = shap.GradientExplainer(self.model, bg_sample)
 
 else:
 # KernelExplainer ( , )
 bg_sample = self.background_data
 if len(bg_sample) > self.max_samples_for_kernel:
 bg_sample = shap.sample(bg_sample, self.max_samples_for_kernel)
 
 if hasattr(self.model, 'predict_proba'):
 predict_fn = lambda x: self.model.predict_proba(x)[:, 1]
 else:
 predict_fn = self.model.predict
 
 self.explainer = shap.KernelExplainer(predict_fn, bg_sample)
 
 return self
 
 def explain(
 self,
 X: Union[np.ndarray, pd.DataFrame],
 check_additivity: bool = False,
 ) -> shap.Explanation:
 """
 Calculate SHAP values
 
 Parameters
 ----------
 X : array-like
 Instance to explain
 check_additivity : bool
 Whether to perform additivity check (tree models)
 """
 
 if self.explainer is None:
 raise ValueError("fit() .")
 
 if isinstance(X, pd.DataFrame):
 X = X.values
 
 if self.explainer_type == 'tree':
 shap_values = self.explainer.shap_values(
 X, check_additivity=check_additivity
 )
 else:
 shap_values = self.explainer.shap_values(X)
 
 # Explanation Generation
 if isinstance(shap_values, list):
 # : 1 ( ) 
 shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
 
 return shap.Explanation(
 values=shap_values,
 base_values=self.explainer.expected_value if not isinstance(
 self.explainer.expected_value, list
 ) else self.explainer.expected_value[1],
 data=X,
 feature_names=self.feature_names,
 )
 
 def explain_single(
 self,
 instance: Union[np.ndarray, pd.Series],
 top_k: int = 5,
 ) -> Dict:
 """
 Explain a single instance
 
 Returns
 -------
 dict
 {
 'shap_values': {feature: value},
 'top_positive': [(feature, value), ...],
 'top_negative': [(feature, value), ...],
 'base_value': float,
 'prediction': float,
 }
 """
 
 if isinstance(instance, pd.Series):
 instance = instance.values
 
 instance = instance.reshape(1, -1)
 explanation = self.explain(instance)
 
 shap_dict = {}
 for i, feat in enumerate(self.feature_names or range(len(explanation.values[0]))):
 shap_dict[feat] = explanation.values[0][i]
 
 # Alignment
 sorted_positive = sorted(
 [(k, v) for k, v in shap_dict.items() if v > 0],
 key=lambda x: x[1], reverse=True
 )[:top_k]
 
 sorted_negative = sorted(
 [(k, v) for k, v in shap_dict.items() if v < 0],
 key=lambda x: x[1]
 )[:top_k]
 
 return {
 'shap_values': shap_dict,
 'top_positive': sorted_positive,
 'top_negative': sorted_negative,
 'base_value': float(explanation.base_values),
 'prediction': float(explanation.base_values + explanation.values[0].sum()),
 }
 
 def global_importance(
 self,
 X: Union[np.ndarray, pd.DataFrame],
 method: str = 'mean_abs',
 ) -> pd.DataFrame:
 """
 Calculate global feature importance
 
 Parameters
 ----------
 method : str
 'mean_abs': Mean absolute value
 'mean': Mean (with direction)
 'max_abs': Max absolute value
 """
 
 explanation = self.explain(X)
 
 if method == 'mean_abs':
 importance = np.abs(explanation.values).mean(axis=0)
 elif method == 'mean':
 importance = explanation.values.mean(axis=0)
 elif method == 'max_abs':
 importance = np.abs(explanation.values).max(axis=0)
 else:
 raise ValueError(f"Unknown method: {method}")
 
 df = pd.DataFrame({
 'feature': self.feature_names or [f'feature_{i}' for i in range(len(importance))],
 'importance': importance,
 })
 
 return df.sort_values('importance', ascending=False).reset_index(drop=True)
 
 def generate_governance_explanation(
 self,
 instance: np.ndarray,
 audience: str = 'technical',
 feature_descriptions: Optional[Dict[str, str]] = None,
 ) -> str:
 """
 Generate explanation text for governance
 
 Parameters
 ----------
 audience : str
 'end_user', 'business', 'technical', 'regulator'
 """
 
 result = self.explain_single(instance)
 
 if audience == 'end_user':
 # For end users (Simplified)
 text = "## Key Decision Factors\n\n"
 text += "The following factors influenced the decision:\n\n"
 
 text += "### Impact\n"
 for feat, val in result['top_positive'][:3]:
 desc = feature_descriptions.get(feat, feat) if feature_descriptions else feat
 text += f"- {desc}\n"
 
 if result['top_negative']:
 text += "\n### Impact\n"
 for feat, val in result['top_negative'][:3]:
 desc = feature_descriptions.get(feat, feat) if feature_descriptions else feat
 text += f"- {desc}\n"
 
 elif audience == 'technical':
 # For technical teams (Detailed metrics)
 text = "## SHAP Analysis Results\n\n"
 text += f"- Base Value: {result['base_value']:.4f}\n"
 text += f"- Prediction: {result['prediction']:.4f}\n\n"
 
 text += "### SHAP Values (Top 10)\n\n"
 text += "| Feature | SHAP Value | Direction |\n"
 text += "|---------|------------|-----------|--\n"
 
 sorted_all = sorted(
 result['shap_values'].items(),
 key=lambda x: abs(x[1]), reverse=True
 )[:10]
 
 for feat, val in sorted_all:
 direction = "" if val > 0 else ""
 text += f"| {feat} | {val:.4f} | {direction} |\n"
 
 elif audience == 'regulator':
 # For regulators (Compliance with Enforcement Decree)
 text = "## AI Decision Explanation (Article 23)\n\n"
 text += f"### Prediction Result\n"
 text += f"- Base Value: {result['base_value']:.4f}\n"
 text += f"- Final Prediction: {result['prediction']:.4f}\n\n"
 
 text += "### Major Impact ( 5 )\n\n"
 text += "| | | | Impact |\n"
 text += "|------|------|--------|----------|\n"
 
 sorted_all = sorted(
 result['shap_values'].items(),
 key=lambda x: abs(x[1]), reverse=True
 )[:5]
 
 for i, (feat, val) in enumerate(sorted_all, 1):
 direction = " " if val > 0 else " "
 text += f"| {i} | {feat} | {val:.4f} | {direction} |\n"
 
 text += "\n*This explanation is based on SHAP (SHapley Additive exPlanations) methodology.*\n"
 text += "*Retained for 5 years per Article 15.*\n"
 
 else: # business
 text = "## Business Insights\n\n"
 text += "### Key factors influencing the decision\n\n"
 
 for i, (feat, val) in enumerate(result['top_positive'][:3], 1):
 desc = feature_descriptions.get(feat, feat) if feature_descriptions else feat
 text += f"{i}. {desc} (+{val:.2f})\n"
 
 return text

### 딥러닝 모델을 위한 설명가능한 AI(설명가능한 AI(XAI)): Captum

**Captum 0.7.0Advanced Deep Learning with XAI**

In [ ]:
"""
Captum 0.7.0 Usage Guide

- Integrated Gradients
- Layer GradCAM (Vision)
- Attention Visualization
"""

try:
 import torch
 import torch.nn as nn
 import numpy as np
 from typing import Optional, Tuple, List
 from captum.attr import (
 IntegratedGradients,
 LayerGradCam,
 GuidedGradCam,
 Saliency,
 DeepLift,
 NoiseTunnel,
 visualization as viz,
 )

 class DeepLearningExplainer:
 """
 Captum-based deep learning model explainer
 """
 
 def __init__(
 self,
 model: nn.Module,
 device: str = 'cpu',
 ):
 self.model = model.to(device)
 self.model.eval()
 self.device = device
 
 def integrated_gradients(
 self,
 inputs: torch.Tensor,
 target: int,
 baseline: Optional[torch.Tensor] = None,
 n_steps: int = 50,
 ) -> torch.Tensor:
 """
 Calculate Integrated Gradients
 
 Reference: Sundararajan et al., "Axiomatic Attribution for Deep Networks" (2017)
 """
 
 ig = IntegratedGradients(self.model)
 
 inputs = inputs.to(self.device)
 if baseline is None:
 baseline = torch.zeros_like(inputs).to(self.device)
 else:
 baseline = baseline.to(self.device)
 
 attributions = ig.attribute(
 inputs,
 baselines=baseline,
 target=target,
 n_steps=n_steps,
 return_convergence_delta=False,
 )
 
 return attributions
 
 def grad_cam(
 self,
 inputs: torch.Tensor,
 target: int,
 layer: nn.Module,
 ) -> torch.Tensor:
 """
 Grad-CAM for CNN
 
 Visualize activation maps for Vision models
 """
 
 grad_cam = LayerGradCam(self.model, layer)
 
 inputs = inputs.to(self.device)
 attributions = grad_cam.attribute(inputs, target=target)
 
 return attributions
 
 def explain_image_classification(
 self,
 image: torch.Tensor,
 target_class: int,
 conv_layer: nn.Module,
 class_names: Optional[List[str]] = None,
 ) -> dict:
 """
 Comprehensive explanation for image classification models
 """
 
 image = image.to(self.device)
 
 with torch.no_grad():
 output = self.model(image)
 probs = torch.softmax(output, dim=1)
 predicted_class = probs.argmax(dim=1).item()
 confidence = probs[0, predicted_class].item()
 
 # Integrated Gradients
 ig_attr = self.integrated_gradients(image, target_class)
 
 # Grad-CAM
 gradcam_attr = self.grad_cam(image, target_class, conv_layer)
 
 return {
 'predicted_class': predicted_class,
 'predicted_label': class_names[predicted_class] if class_names else str(predicted_class),
 'confidence': confidence,
 'target_class': target_class,
 'integrated_gradients': ig_attr.cpu().numpy(),
 'grad_cam': gradcam_attr.cpu().numpy(),
 }
 
 def explain_tabular(
 self,
 inputs: torch.Tensor,
 target: int,
 feature_names: List[str],
 method: str = 'integrated_gradients',
 ) -> dict:
 """
 Data Model
 """
 
 inputs = inputs.to(self.device)
 
 if method == 'integrated_gradients':
 ig = IntegratedGradients(self.model)
 attributions = ig.attribute(inputs, target=target)
 elif method == 'deeplift':
 dl = DeepLift(self.model)
 attributions = dl.attribute(inputs, target=target)
 elif method == 'saliency':
 sal = Saliency(self.model)
 attributions = sal.attribute(inputs, target=target)
 else:
 raise ValueError(f"Unknown method: {method}")
 
 attr_values = attributions.cpu().numpy().flatten()
 
 feature_importance = {
 feat: float(val)
 for feat, val in zip(feature_names, attr_values)
 }
 
 # Alignment
 sorted_importance = sorted(
 feature_importance.items(),
 key=lambda x: abs(x[1]),
 reverse=True
 )
 
 return {
 'method': method,
 'feature_importance': feature_importance,
 'sorted_importance': sorted_importance,
 'top_positive': [(k, v) for k, v in sorted_importance if v > 0][:5],
 'top_negative': [(k, v) for k, v in sorted_importance if v < 0][:5],
 }

except ImportError:
 print("captum 미설치. !pip install captum 후 재실행하세요.")

## Model 관리 및 MLOps 통합된

### MLflow와 거버넌스 통합된

**MLflow Governance Integration**

In [ ]:
"""
MLflow + AI Governance Integration

- Fairness Metric Logging
- Model Card Integration
- Approval Status Management
- Governance Tags
"""

import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models import ModelSignature
from typing import Dict, Optional, List, Any
from datetime import datetime
import json

class MLflowGovernanceIntegration:
 """
 MLflow AI Governance Integrated
 
 Tracking, Model Governance information Integration
 """
 
 # Governance Tags 
 GOVERNANCE_PREFIX = "governance"
 
 # Governance Tags
 GOVERNANCE_TAGS = {
 'risk_level': f'{GOVERNANCE_PREFIX}.risk_level',
 'high_risk_sector': f'{GOVERNANCE_PREFIX}.high_risk_sector',
 'fairness_status': f'{GOVERNANCE_PREFIX}.fairness_status',
 'approval_status': f'{GOVERNANCE_PREFIX}.approval_status',
 'approved_by': f'{GOVERNANCE_PREFIX}.approved_by',
 'approval_date': f'{GOVERNANCE_PREFIX}.approval_date',
 'human_oversight_level': f'{GOVERNANCE_PREFIX}.human_oversight_level',
 'retention_until': f'{GOVERNANCE_PREFIX}.retention_until',
 }
 
 def __init__(
 self,
 tracking_uri: Optional[str] = None,
 experiment_name: Optional[str] = None,
 ):
 if tracking_uri:
 mlflow.set_tracking_uri(tracking_uri)
 
 if experiment_name:
 mlflow.set_experiment(experiment_name)
 
 self.client = MlflowClient()
 
 def log_fairness_metrics(
 self,
 fairness_results: Dict[str, float],
 step: Optional[int] = None,
 ) -> None:
 """Fairness Metric Logging"""
 
 for metric_name, value in fairness_results.items():
 mlflow.log_metric(
 f"fairness.{metric_name}",
 value,
 step=step
 )
 
 # 4/5 dp_ratio = fairness_results.get('dp_ratio', 0)
 passes_4_5 = dp_ratio >= 0.8
 mlflow.log_metric("fairness.passes_4_5_rule", int(passes_4_5))
 
 def log_governance_tags(
 self,
 risk_level: str,
 high_risk_sector: Optional[str] = None,
 fairness_status: str = 'pending',
 human_oversight_level: str = 'HOTL',
 ) -> None:
 """Governance Tags Logging"""
 
 tags = {
 self.GOVERNANCE_TAGS['risk_level']: risk_level,
 self.GOVERNANCE_TAGS['fairness_status']: fairness_status,
 self.GOVERNANCE_TAGS['human_oversight_level']: human_oversight_level,
 }
 
 if high_risk_sector:
 tags[self.GOVERNANCE_TAGS['high_risk_sector']] = high_risk_sector
 
 # 5years Calculation
 retention_until = datetime.now().replace(year=datetime.now().year + 5)
 tags[self.GOVERNANCE_TAGS['retention_until']] = retention_until.isoformat()
 
 mlflow.set_tags(tags)
 
 def log_model_card(
 self,
 model_card: Dict[str, Any],
 artifact_path: str = "governance/model_card.json",
 ) -> None:
 """Model """
 
 # JSON with open("/tmp/model_card.json", "w", encoding="utf-8") as f:
 json.dump(model_card, f, ensure_ascii=False, indent=2, default=str)
 
 mlflow.log_artifact("/tmp/model_card.json", artifact_path="governance")
 
 def log_fairness_report(
 self,
 report_content: str,
 report_format: str = "md",
 ) -> None:
 """Fairness Report """
 
 filename = f"/tmp/fairness_report.{report_format}"
 with open(filename, "w", encoding="utf-8") as f:
 f.write(report_content)
 
 mlflow.log_artifact(filename, artifact_path="governance")
 
 def register_model_with_governance(
 self,
 model_uri: str,
 name: str,
 risk_level: str,
 fairness_metrics: Dict[str, float],
 approval_status: str = 'pending',
 tags: Optional[Dict[str, str]] = None,
 ) -> str:
 """Governance information Model """
 
 # Model 
 result = mlflow.register_model(model_uri, name)
 model_version = result.version
 
 # Governance Tags Setup
 governance_tags = {
 self.GOVERNANCE_TAGS['risk_level']: risk_level,
 self.GOVERNANCE_TAGS['approval_status']: approval_status,
 self.GOVERNANCE_TAGS['fairness_status']: 'pass' if fairness_metrics.get('dp_ratio', 0) >= 0.8 else 'fail',
 }
 
 if tags:
 governance_tags.update(tags)
 
 # Model Version Tags Setup
 for key, value in governance_tags.items():
 self.client.set_model_version_tag(name, model_version, key, value)
 
 # Fairness (description) Addition
 description = f"""
## Governance Information

- Risk Level: {risk_level}
- Approval Status: {approval_status}
- DP Ratio: {fairness_metrics.get('dp_ratio', 'N/A')}
- EO Diff: {fairness_metrics.get('eo_diff', 'N/A')}
- 4/5 Rule: {'Pass' if fairness_metrics.get('dp_ratio', 0) >= 0.8 else 'Fail'}
"""
 
 self.client.update_model_version(
 name=name,
 version=model_version,
 description=description
 )
 
 return model_version
 
 def approve_model_version(
 self,
 name: str,
 version: str,
 approver: str,
 conditions: Optional[List[str]] = None,
 ) -> None:
 """Model Version Approval"""
 
 approval_date = datetime.now().isoformat()
 
 self.client.set_model_version_tag(
 name, version,
 self.GOVERNANCE_TAGS['approval_status'], 'approved'
 )
 self.client.set_model_version_tag(
 name, version,
 self.GOVERNANCE_TAGS['approved_by'], approver
 )
 self.client.set_model_version_tag(
 name, version,
 self.GOVERNANCE_TAGS['approval_date'], approval_date
 )
 
 if conditions:
 self.client.set_model_version_tag(
 name, version,
 f"{self.GOVERNANCE_PREFIX}.approval_conditions",
 json.dumps(conditions)
 )
 
 # (Staging -> Production)
 self.client.transition_model_version_stage(
 name=name,
 version=version,
 stage="Production",
 archive_existing_versions=True
 )
 
 def get_governance_status(
 self,
 name: str,
 version: Optional[str] = None,
 ) -> Dict:
 """Model Governance Status """
 
 if version is None:
 # Version 
 versions = self.client.search_model_versions(f"name='{name}'")
 if not versions:
 raise ValueError(f"Model {name} not found")
 version = max(v.version for v in versions)
 
 model_version = self.client.get_model_version(name, version)
 
 governance_info = {}
 for tag_name, tag_key in self.GOVERNANCE_TAGS.items():
 if tag_key in model_version.tags:
 governance_info[tag_name] = model_version.tags[tag_key]
 
 return {
 'model_name': name,
 'version': version,
 'stage': model_version.current_stage,
 'governance': governance_info,
 }

### CI/CD 파이프라인 통합된

## 모니터링 및 관측성 플랫폼

### Evidently AI를 활용한 모니터링

**Evidently AI Monitoring Integration**

In [ ]:
"""
Evidently AI Monitoring Setup

- Data Drift Detection
- Performance Monitoring
- Fairness Tracking
- Automated Report Generation
"""

import pandas as pd
import numpy as np
from datetime import datetime
from typing import Optional, Dict, List

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import (
 DataDriftPreset,
 DataQualityPreset,
 ClassificationPreset,
)
from evidently.metrics import (
 DataDriftTable,
 DatasetDriftMetric,
 ColumnDriftMetric,
 ClassificationQualityMetric,
 ClassificationClassBalance,
)
from evidently.test_suite import TestSuite
from evidently.test_preset import (
 DataDriftTestPreset,
 DataQualityTestPreset,
)
from evidently.tests import (
 TestColumnDrift,
 TestShareOfDriftedColumns,
 TestAccuracyScore,
)

class EvidentlyGovernanceMonitor:
 """
 Evidently AI based Governance Monitoring
 """
 
 def __init__(
 self,
 reference_data: pd.DataFrame,
 column_mapping: Optional[ColumnMapping] = None,
 ):
 """
 Parameters
 ----------
 reference_data : pd.DataFrame
 Data (Learning Data Data)
 column_mapping : ColumnMapping
 (target, prediction, numerical, categorical )
 """
 self.reference_data = reference_data
 self.column_mapping = column_mapping or ColumnMapping()
 
 def create_drift_report(
 self,
 current_data: pd.DataFrame,
 output_path: Optional[str] = None,
 ) -> Report:
 """Data Drift Report Generation"""
 
 report = Report(metrics=[
 DataDriftPreset(),
 DatasetDriftMetric(),
 ])
 
 report.run(
 reference_data=self.reference_data,
 current_data=current_data,
 column_mapping=self.column_mapping,
 )
 
 if output_path:
 report.save_html(output_path)
 
 return report
 
 def create_performance_report(
 self,
 current_data: pd.DataFrame,
 output_path: Optional[str] = None,
 ) -> Report:
 """Performance Monitoring Report Generation"""
 
 report = Report(metrics=[
 ClassificationPreset(),
 ClassificationClassBalance(),
 ])
 
 report.run(
 reference_data=self.reference_data,
 current_data=current_data,
 column_mapping=self.column_mapping,
 )
 
 if output_path:
 report.save_html(output_path)
 
 return report
 
 def run_drift_tests(
 self,
 current_data: pd.DataFrame,
 drift_threshold: float = 0.1,
 ) -> Dict:
 """Drift """
 
 test_suite = TestSuite(tests=[
 TestShareOfDriftedColumns(lt=drift_threshold),
 DataDriftTestPreset(),
 ])
 
 test_suite.run(
 reference_data=self.reference_data,
 current_data=current_data,
 column_mapping=self.column_mapping,
 )
 
 results = {
 'passed': test_suite.as_dict()['summary']['all_passed'],
 'total_tests': test_suite.as_dict()['summary']['total_tests'],
 'failed_tests': test_suite.as_dict()['summary']['failed_tests'],
 'details': test_suite.as_dict()['tests'],
 }
 
 return results
 
 def monitor_fairness_over_time(
 self,
 data_batches: List[pd.DataFrame],
 sensitive_column: str,
 prediction_column: str = 'prediction',
 target_column: str = 'target',
 ) -> pd.DataFrame:
 """
 Fairness Tracking
 
 Data Fairness Calculation
 """
 
 results = []
 
 for i, batch in enumerate(data_batches):
 groups = batch[sensitive_column].unique()
 
 # Calculation
 selection_rates = {}
 for group in groups:
 mask = batch[sensitive_column] == group
 selection_rates[group] = batch.loc[mask, prediction_column].mean()
 
 # DP Ratio Calculation
 if len(selection_rates) >= 2:
 rates = list(selection_rates.values())
 dp_ratio = min(rates) / max(rates) if max(rates) > 0 else 1.0
 else:
 dp_ratio = 1.0
 
 row = {
 'batch': i,
 'dp_ratio': dp_ratio,
 'passes_4_5_rule': dp_ratio >= 0.8,
 }
 for g, r in selection_rates.items():
 row[f'selection_rate_{g}'] = r
 results.append(row)
 
 return pd.DataFrame(results)
 
 def generate_governance_dashboard_data(
 self,
 current_data: pd.DataFrame,
 ) -> Dict:
 """Governance Data Generation"""
 
 # Drift Analysis
 drift_report = self.create_drift_report(current_data)
 drift_results = drift_report.as_dict()
 
 # Performance Analysis
 perf_report = self.create_performance_report(current_data)
 perf_results = perf_report.as_dict()
 
 # Data 
 dashboard_data = {
 'timestamp': datetime.now().isoformat(),
 'data_quality': {
 'total_rows': len(current_data),
 'missing_rate': current_data.isnull().mean().mean(),
 },
 'drift': {
 'dataset_drift': drift_results.get('metrics', [{}])[0].get('result', {}),
 'drifted_columns': [],
 },
 'performance': {
 'metrics': perf_results.get('metrics', [{}])[0].get('result', {}),
 },
 'alerts': [],
 }
 
 # Generation
 if dashboard_data['drift'].get('dataset_drift', {}).get('drift_share', 0) > 0.2:
 dashboard_data['alerts'].append({
 'severity': 'warning',
 'message': 'Data Drift 20% .',
 })
 
 return dashboard_data

## 문서화 자동화 도구

### Model Card Toolkit

**Integrated Documentation Automation System**

In [ ]:
"""
AI Governance Documentation Automation

- Automated Model Card Generation
- Fairness Report Automation
- Compliance Document Package
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
from datetime import datetime
import json
import os

@dataclass
class GovernanceDocumentPackage:
 """
 AI Governance Document 15 EU AI Act Annex IV Compliance Document Generation
 """
 
 # System information
 system_name: str
 system_id: str
 version: str
 developer: str
 
 # Classification
 risk_level: str # 'high', 'medium', 'low'
 high_risk_sector: Optional[str] = None
 
 # Model information
 model_type: str = ""
 intended_use: List[str] = field(default_factory=list)
 out_of_scope_use: List[str] = field(default_factory=list)
 
 # Data information
 training_data_source: str = ""
 training_data_size: int = 0
 training_data_period: str = ""
 
 # Performance Fairness
 performance_metrics: Dict[str, float] = field(default_factory=dict)
 fairness_metrics: Dict[str, float] = field(default_factory=dict)
 subgroup_performance: Dict[str, Dict[str, float]] = field(default_factory=dict)
 
 # 
 xai_method: str = ""
 feature_importance: Dict[str, float] = field(default_factory=dict)
 
 human_oversight_level: str = ""
 override_rate: float = 0.0
 
 identified_risks: List[Dict] = field(default_factory=list)
 mitigation_measures: List[str] = field(default_factory=list)
 
 # Data
 creation_date: datetime = field(default_factory=datetime.now)
 retention_years: int = 5
 
 def generate_model_card(self) -> str:
 """Model Generation ( )"""
 
 card = f"""# Model : {self.system_name}

## 1. Model 

| Items | |
|------|------|
| System | {self.system_name} |
| System ID | {self.system_id} |
| Version | {self.version} |
| | {self.developer} |
| Model | {self.model_type} |
| Level | {self.risk_level.upper()} |
"""
 
 if self.high_risk_sector:
 card += f"| | {self.high_risk_sector} |\n"
 
 card += f"""
## 2. ### Case
"""
 for use in self.intended_use:
 card += f"- {use}\n"
 
 card += "\n### ( )\n"
 for use in self.out_of_scope_use:
 card += f"- {use}\n"
 
 card += f"""

## 3. Learning Data

| Items | |
|------|------|
| Data | {self.training_data_source} |
| Data | {self.training_data_size:,} |
| Period | {self.training_data_period} |

## 4. Performance

| | |
|--------|-----|
"""
 for metric, value in self.performance_metrics.items():
 card += f"| {metric} | {value:.4f} |\n"
 
 card += f"""

## 5. Fairness Assessment

| | | |
|--------|-----|----------|
"""
 for metric, value in self.fairness_metrics.items():
 passed = '[PASS]' if metric == 'dp_ratio' and value >= 0.8 else '[PASS]' if metric == 'eo_diff' and value <= 0.1 else '[WARN]'
 card += f"| {metric} | {value:.4f} | {passed} |\n"
 
 if self.subgroup_performance:
 card += "\n### Performance\n\n"
 card += "| | Accuracy | Precision | Recall |\n"
 card += "|------|----------|-----------|--------|\n"
 for group, metrics in self.subgroup_performance.items():
 card += f"| {group} | {metrics.get('accuracy', 0):.4f} | {metrics.get('precision', 0):.4f} | {metrics.get('recall', 0):.4f} |\n"
 
 card += f"""

## 6. 

-XAI : {self.xai_method}

### Major ( 10 )

| | | |
|------|------|--------|
"""
 sorted_importance = sorted(
 self.feature_importance.items(),
 key=lambda x: abs(x[1]),
 reverse=True
 )[:10]
 for i, (feat, imp) in enumerate(sorted_importance, 1):
 card += f"| {i} | {feat} | {imp:.4f} |\n"
 
 card += f"""

## 7. | Items | |
|------|------|
| Level | {self.human_oversight_level} |
| AI | {self.override_rate:.2%} |

## 8. ### """
 for risk in self.identified_risks:
 card += f"-{risk.get('name', 'N/A')}** ( : {risk.get('severity', 'N/A')}): {risk.get('description', '')}\n"
 
 card += "\n### \n"
 for measure in self.mitigation_measures:
 card += f"- {measure}\n"
 
 card += f"""

---

## Document information

| Items | |
|------|------|
| days | {self.creation_date.strftime('%Y-%m-%d')} |
| | {self.creation_date.year + self.retention_years}years ({self.retention_years}years) |

* Document AI 15 .*
"""
 
 return card
 
 def generate_compliance_checklist(self) -> str:
 """Regulation Compliance Checklist Generation"""
 
 checklist = f"""# Regulation Compliance Checklist

System: {self.system_name} ({self.system_id}) 
Assessmentdays: {datetime.now().strftime('%Y-%m-%d')}

---

## AI Compliance

| | | Compliance | |
|------|----------|----------|------|
| 10 | AI Classification | {'[PASS]' if self.risk_level == 'high' and self.high_risk_sector else '[PASS]' if self.risk_level != 'high' else ''} | Classification Document |
| 12 | | {'[PASS]' if self.fairness_metrics.get('dp_ratio', 0) >= 0.8 else '[FAIL]'} | Fairness Assessment Report |
| 13 | | | |
| 14 | | {'[PASS]' if self.human_oversight_level in ['HITL', 'HOTL'] else '[WARN]'} | Document |
| 15 | (5years) | [PASS] | Document |
| 23 | | {'[PASS]' if self.xai_method else '[FAIL]'} | Model , XAI Report |

---

## EU AI Act Compliance ( )

| | | Compliance |
|------|----------|----------|
| Art. 9 | Management System | {'[PASS]' if self.identified_risks else '[FAIL]'} |
| Art. 10 | Data Governance | |
| Art. 11 | Document | [PASS] |
| Art. 13 | | [PASS] |
| Art. 14 | | {'[PASS]' if self.human_oversight_level in ['HITL', 'HOTL'] else '[WARN]'} |

---

Check: ________________ 
days: ________________ 
 : ________________
"""
 
 return checklist
 
 def export_package(self, output_dir: str) -> List[str]:
 """ Document """
 
 os.makedirs(output_dir, exist_ok=True)
 
 files_created = []
 
 # 1. Model 
 model_card_path = os.path.join(output_dir, "model_card.md")
 with open(model_card_path, "w", encoding="utf-8") as f:
 f.write(self.generate_model_card())
 files_created.append(model_card_path)
 
 # 2. Compliance Checklist
 checklist_path = os.path.join(output_dir, "compliance_checklist.md")
 with open(checklist_path, "w", encoding="utf-8") as f:
 f.write(self.generate_compliance_checklist())
 files_created.append(checklist_path)
 
 # 3. Data JSON
 metadata = {
 "system_name": self.system_name,
 "system_id": self.system_id,
 "version": self.version,
 "risk_level": self.risk_level,
 "high_risk_sector": self.high_risk_sector,
 "fairness_metrics": self.fairness_metrics,
 "performance_metrics": self.performance_metrics,
 "human_oversight_level": self.human_oversight_level,
 "creation_date": self.creation_date.isoformat(),
 "retention_until": (self.creation_date.year + self.retention_years),
 }
 
 metadata_path = os.path.join(output_dir, "metadata.json")
 with open(metadata_path, "w", encoding="utf-8") as f:
 json.dump(metadata, f, ensure_ascii=False, indent=2)
 files_created.append(metadata_path)
 
 return files_created

### 제6장 요약